# 1. Models through the dedicated AI Gateway

An application sends a gateway **runtime** key and a **callable model alias**, not a Foundry credential or an APIM subscription key. All three notebooks share the same chat endpoint, runtime key, and model alias. If you do not already have `.env`, copy `.env.example` to `.env` in the repository root and fill in those three settings using the [AI Gateway portal](https://ai.gateway.azure.com); see the repository README for where to find them. An alias can target one model or a pool. Shell environment variables override `.env`; never save keys or request headers in notebook output.

**Expected result:** After running the cells from top to bottom, the output includes the gateway host, selected alias, a model response, and reported token usage, if present. One run sends one request; rerunning the request sends another.

In [ ]:
import os
from pathlib import Path
from urllib.parse import urlparse

import requests
from dotenv import dotenv_values

notebook_path = globals().get('__vsc_ipynb_file__')
search_roots = [Path.cwd()]
if notebook_path:
    search_roots.insert(0, Path(notebook_path).parent)
lab_root = next(
    (directory for start in search_roots for directory in (start, *start.parents)
     if (directory / '.env.example').is_file() and (directory / 'notebooks').is_dir()),
    Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd(),
)
env_path = lab_root / '.env'
file_settings = dotenv_values(env_path, encoding='utf-8-sig') if env_path.exists() else {}
required = ('AI_GATEWAY_CHAT_URL', 'AI_GATEWAY_API_KEY', 'AI_GATEWAY_MODEL_ALIAS')
settings = {name: (os.environ[name] if name in os.environ else (file_settings.get(name) or '')).strip()
            for name in required}

chat_url, api_key, alias = (settings[name] for name in required)
missing = [name for name in required if not settings[name]]
del file_settings, settings
if missing:
    status = 'found' if env_path.is_file() else 'not found'
    blank_overrides = [name for name in missing if name in os.environ]
    hint = (f" Blank kernel environment variables override .env: {', '.join(blank_overrides)}."
            ' Remove them or restart the kernel after changing the launching environment.'
            if blank_overrides else '')
    raise RuntimeError(f"Missing or blank settings: {', '.join(missing)}. "
                       f'Checked {env_path} ({status}). Set the missing values there or in the kernel environment.' + hint)
parsed = urlparse(chat_url)
if parsed.scheme != 'https' or not parsed.netloc or not parsed.path.endswith('/chat/completions'):
    raise ValueError('Use the full HTTPS Chat completions endpoint copied from the model Overview in AI Gateway.')
print('Gateway host:', parsed.hostname, '| model alias:', alias)


## A single OpenAI-compatible request

The gateway authenticates the caller and uses its configured backend authentication, such as managed identity to Foundry. The request uses the stable **alias** rather than a backend deployment URL.

In [ ]:
def chat(prompt, model=alias):
    response = requests.post(
        chat_url,
        headers={'Authorization': f'Bearer {api_key}', 'Content-Type': 'application/json'},
        json={'model': model, 'messages': [{'role': 'user', 'content': prompt}]},
        timeout=90,
    )
    response.raise_for_status()
    return response.json()

result = chat('In one sentence, what does a model gateway do?')
print('Response:', result['choices'][0]['message']['content'])
print('Reported usage:', result.get('usage', 'not included by this model'))


## Stable model aliases

In the portal, inspect your alias and its registered backend model targets. As an optional exercise, change a target on an isolated **non-production** alias and rerun the previous request without changing application code. The callable name stays the same while its backend mapping changes. Changing a shared alias affects other callers. Repeated answers alone do not prove which backend was selected; use backend telemetry. To compare with a separate single-model alias instead, update `AI_GATEWAY_MODEL_ALIAS` in `.env` and restart the kernel; shell overrides take precedence.